## tl;dr

**READY FOR SEGMENTED RESEARCH** as of `2026-08-03T18:00:00+00:00`.

- 362,774 candles profiled across two pairs and two research timeframes.
- Zero validity failures: no duplicate timestamps, null cells, negative values, or impossible OHLC relationships.
- Zero 15m-to-1h reconciliation mismatches on complete hours.
- A historical listing/trading gap exists; the largest gap contains 15,758 missing `15m` candles for `BTC/USDC`.
- The weakest pre-declared research split still has 100.0000% coverage.


## Context & Methods

This notebook validates the local Binance Spot OHLCV extracts used by the Freqtrade research environment.

### Key Assumptions

- Grain: one UTC exchange candle per `pair × timeframe × candle-open timestamp`.
- Inputs: Binance public OHLCV data downloaded by Freqtrade 2026.7 / CCXT 4.5.68.
- Intended use: segmented backtesting for BTC/USDC and ETH/USDC, not tick-level execution modeling.
- A long exchange/listing gap blocks full-period backtests but does not invalidate later contiguous segments.
- The last still-forming candle is excluded from freshness conclusions.


## Data

### 1. Load the local Feather extracts

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "user_data").exists():
    raise RuntimeError(f"Execute from the project root, got: {PROJECT_ROOT}")

DATA_DIR = PROJECT_ROOT / "user_data" / "data" / "binance"
REPORT_DIR = PROJECT_ROOT / "user_data" / "reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

PAIRS = ("BTC_USDC", "ETH_USDC")
TIMEFRAMES = {"15m": pd.Timedelta(minutes=15), "1h": pd.Timedelta(hours=1)}
frames = {}

for pair in PAIRS:
    for timeframe in TIMEFRAMES:
        path = DATA_DIR / f"{pair}-{timeframe}.feather"
        frame = pd.read_feather(path).sort_values("date").reset_index(drop=True)
        frame["date"] = pd.to_datetime(frame["date"], utc=True)
        frames[(pair, timeframe)] = frame

{f"{pair}-{timeframe}": list(frame.columns) for (pair, timeframe), frame in frames.items()}

{'BTC_USDC-15m': ['date', 'open', 'high', 'low', 'close', 'volume'],
 'BTC_USDC-1h': ['date', 'open', 'high', 'low', 'close', 'volume'],
 'ETH_USDC-15m': ['date', 'open', 'high', 'low', 'close', 'volume'],
 'ETH_USDC-1h': ['date', 'open', 'high', 'low', 'close', 'volume']}

## Results

### 2. Profile grain, completeness, and domain validity

In [2]:
def profile_frame(pair, timeframe, frame):
    interval = TIMEFRAMES[timeframe]
    unique_dates = frame["date"].drop_duplicates().sort_values()
    expected_rows = int((unique_dates.iloc[-1] - unique_dates.iloc[0]) / interval) + 1
    duplicate_timestamps = int(frame["date"].duplicated().sum())
    null_cells = int(frame.isna().sum().sum())
    invalid_prices = int((frame[["open", "high", "low", "close"]] <= 0).any(axis=1).sum())
    invalid_volume = int((frame["volume"] < 0).sum())
    invalid_ohlc = int((
        (frame["high"] < frame[["open", "close", "low"]].max(axis=1))
        | (frame["low"] > frame[["open", "close", "high"]].min(axis=1))
    ).sum())
    missing_candles = expected_rows - len(unique_dates)
    return {
        "pair": pair.replace("_", "/"),
        "timeframe": timeframe,
        "rows": len(frame),
        "from_utc": unique_dates.iloc[0],
        "to_utc": unique_dates.iloc[-1],
        "expected_rows": expected_rows,
        "missing_candles": missing_candles,
        "coverage_pct": 100 * len(unique_dates) / expected_rows,
        "duplicate_timestamps": duplicate_timestamps,
        "null_cells": null_cells,
        "invalid_prices": invalid_prices,
        "invalid_volume": invalid_volume,
        "invalid_ohlc": invalid_ohlc,
    }

profile_df = pd.DataFrame([
    profile_frame(pair, timeframe, frame)
    for (pair, timeframe), frame in frames.items()
])
profile_df.to_csv(REPORT_DIR / "data_quality_profile.csv", index=False)
display(profile_df)

,pair,timeframe,rows,from_utc,to_utc,expected_rows,missing_candles,coverage_pct,duplicate_timestamps,null_cells,invalid_prices,invalid_volume,invalid_ohlc
0,BTC/USDC,15m,145110,2022-01-01 00:00:00+00:00,2026-08-03 18:00:00+00:00,160873,15763,90.201588,0,0,0,0,0
1,BTC/USDC,1h,36277,2022-01-01 00:00:00+00:00,2026-08-03 16:00:00+00:00,40217,3940,90.203148,0,0,0,0,0
2,ETH/USDC,15m,145110,2022-01-01 00:00:00+00:00,2026-08-03 18:00:00+00:00,160873,15763,90.201588,0,0,0,0,0
3,ETH/USDC,1h,36277,2022-01-01 00:00:00+00:00,2026-08-03 16:00:00+00:00,40217,3940,90.203148,0,0,0,0,0


### 3. Locate gaps and contiguous research segments

In [3]:
gap_rows = []
segment_rows = []

for (pair, timeframe), frame in frames.items():
    interval = TIMEFRAMES[timeframe]
    dates = frame["date"].drop_duplicates().sort_values().reset_index(drop=True)
    deltas = dates.diff()
    gap_positions = deltas[deltas > interval].index.tolist()

    for position in gap_positions:
        previous_date = dates.iloc[position - 1]
        next_date = dates.iloc[position]
        gap_rows.append({
            "pair": pair.replace("_", "/"),
            "timeframe": timeframe,
            "previous_candle_utc": previous_date,
            "next_candle_utc": next_date,
            "missing_candles": int((next_date - previous_date) / interval) - 1,
        })

    boundaries = [0] + gap_positions + [len(dates)]
    for segment_number, (start_idx, end_idx) in enumerate(zip(boundaries[:-1], boundaries[1:]), start=1):
        segment = dates.iloc[start_idx:end_idx]
        if segment.empty:
            continue
        segment_rows.append({
            "pair": pair.replace("_", "/"),
            "timeframe": timeframe,
            "segment": segment_number,
            "from_utc": segment.iloc[0],
            "to_utc": segment.iloc[-1],
            "candles": len(segment),
            "duration_days": (segment.iloc[-1] - segment.iloc[0]).total_seconds() / 86400,
        })

gaps_df = pd.DataFrame(gap_rows).sort_values("missing_candles", ascending=False)
segments_df = pd.DataFrame(segment_rows).sort_values(
    ["pair", "timeframe", "candles"], ascending=[True, True, False]
)
gaps_df.to_csv(REPORT_DIR / "data_quality_gaps.csv", index=False)
segments_df.to_csv(REPORT_DIR / "data_quality_segments.csv", index=False)

print("Largest gaps:")
display(gaps_df.head(12))
print("Longest segment per pair/timeframe:")
display(segments_df.groupby(["pair", "timeframe"], as_index=False).first())

Largest gaps:


,pair,timeframe,previous_candle_utc,next_candle_utc,missing_candles
0,BTC/USDC,15m,2022-09-29 02:45:00+00:00,2023-03-12 06:30:00+00:00,15758
4,ETH/USDC,15m,2022-09-29 02:45:00+00:00,2023-03-12 06:30:00+00:00,15758
6,ETH/USDC,1h,2022-09-29 02:00:00+00:00,2023-03-12 06:00:00+00:00,3939
2,BTC/USDC,1h,2022-09-29 02:00:00+00:00,2023-03-12 06:00:00+00:00,3939
5,ETH/USDC,15m,2023-03-24 12:30:00+00:00,2023-03-24 14:00:00+00:00,5
1,BTC/USDC,15m,2023-03-24 12:30:00+00:00,2023-03-24 14:00:00+00:00,5
3,BTC/USDC,1h,2023-03-24 12:00:00+00:00,2023-03-24 14:00:00+00:00,1
7,ETH/USDC,1h,2023-03-24 12:00:00+00:00,2023-03-24 14:00:00+00:00,1


Longest segment per pair/timeframe:


,pair,timeframe,segment,from_utc,to_utc,candles,duration_days
0,BTC/USDC,15m,3,2023-03-24 14:00:00+00:00,2026-08-03 18:00:00+00:00,117905,1228.166667
1,BTC/USDC,1h,3,2023-03-24 14:00:00+00:00,2026-08-03 16:00:00+00:00,29475,1228.083333
2,ETH/USDC,15m,3,2023-03-24 14:00:00+00:00,2026-08-03 18:00:00+00:00,117905,1228.166667
3,ETH/USDC,1h,3,2023-03-24 14:00:00+00:00,2026-08-03 16:00:00+00:00,29475,1228.083333


### 4. Reconcile 15-minute candles against stored hourly candles

In [4]:
consistency_rows = []

for pair in PAIRS:
    frame_15m = frames[(pair, "15m")].copy().set_index("date")
    frame_15m["_count"] = 1
    hourly_from_15m = frame_15m.resample("1h", label="left", closed="left").agg({
        "open": "first",
        "high": "max",
        "low": "min",
        "close": "last",
        "volume": "sum",
        "_count": "sum",
    })
    hourly_from_15m = hourly_from_15m[hourly_from_15m["_count"] == 4].drop(columns="_count")
    stored_hourly = frames[(pair, "1h")].set_index("date")[["open", "high", "low", "close", "volume"]]
    joined = hourly_from_15m.join(stored_hourly, how="inner", lsuffix="_from_15m", rsuffix="_stored")

    mismatch_counts = {}
    max_relative_diffs = {}
    for column in ("open", "high", "low", "close", "volume"):
        left = joined[f"{column}_from_15m"].to_numpy()
        right = joined[f"{column}_stored"].to_numpy()
        matches = np.isclose(left, right, rtol=1e-10, atol=1e-10, equal_nan=True)
        mismatch_counts[column] = int((~matches).sum())
        denominator = np.maximum(np.abs(right), 1e-12)
        max_relative_diffs[column] = float(np.max(np.abs(left - right) / denominator)) if len(joined) else np.nan

    consistency_rows.append({
        "pair": pair.replace("_", "/"),
        "complete_hours_compared": len(joined),
        **{f"{column}_mismatches": mismatch_counts[column] for column in mismatch_counts},
        **{f"{column}_max_relative_diff": max_relative_diffs[column] for column in max_relative_diffs},
    })

consistency_df = pd.DataFrame(consistency_rows)
consistency_df.to_csv(REPORT_DIR / "data_quality_timeframe_consistency.csv", index=False)
display(consistency_df)

,pair,complete_hours_compared,open_mismatches,high_mismatches,low_mismatches,close_mismatches,volume_mismatches,open_max_relative_diff,high_max_relative_diff,low_max_relative_diff,close_max_relative_diff,volume_max_relative_diff
0,BTC/USDC,36275,0,0,0,0,0,0.0,0.0,0.0,0.0,2.452366e-16
1,ETH/USDC,36275,0,0,0,0,0,0.0,0.0,0.0,0.0,2.220408e-16


### 5. Verify the pre-declared train, validation, and holdout windows

In [5]:
SPLITS = {
    "train": (pd.Timestamp("2024-04-01", tz="UTC"), pd.Timestamp("2025-01-01", tz="UTC")),
    "validation": (pd.Timestamp("2025-01-01", tz="UTC"), pd.Timestamp("2026-01-01", tz="UTC")),
    "holdout": (pd.Timestamp("2026-01-01", tz="UTC"), pd.Timestamp("2026-08-01", tz="UTC")),
}

split_rows = []
for split_name, (start, end) in SPLITS.items():
    for (pair, timeframe), frame in frames.items():
        interval = TIMEFRAMES[timeframe]
        split_frame = frame[(frame["date"] >= start) & (frame["date"] < end)]
        expected_rows = int((end - start) / interval)
        unique_rows = split_frame["date"].nunique()
        split_rows.append({
            "split": split_name,
            "pair": pair.replace("_", "/"),
            "timeframe": timeframe,
            "from_utc": start,
            "to_utc_exclusive": end,
            "rows": len(split_frame),
            "expected_rows": expected_rows,
            "missing_candles": expected_rows - unique_rows,
            "coverage_pct": 100 * unique_rows / expected_rows,
        })

split_coverage_df = pd.DataFrame(split_rows)
split_coverage_df.to_csv(REPORT_DIR / "data_quality_split_coverage.csv", index=False)
display(split_coverage_df)

,split,pair,timeframe,from_utc,to_utc_exclusive,rows,expected_rows,missing_candles,coverage_pct
0,train,BTC/USDC,15m,2024-04-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,26400,26400,0,100.0
1,train,BTC/USDC,1h,2024-04-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,6600,6600,0,100.0
2,train,ETH/USDC,15m,2024-04-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,26400,26400,0,100.0
3,train,ETH/USDC,1h,2024-04-01 00:00:00+00:00,2025-01-01 00:00:00+00:00,6600,6600,0,100.0
4,validation,BTC/USDC,15m,2025-01-01 00:00:00+00:00,2026-01-01 00:00:00+00:00,35040,35040,0,100.0
5,validation,BTC/USDC,1h,2025-01-01 00:00:00+00:00,2026-01-01 00:00:00+00:00,8760,8760,0,100.0
6,validation,ETH/USDC,15m,2025-01-01 00:00:00+00:00,2026-01-01 00:00:00+00:00,35040,35040,0,100.0
7,validation,ETH/USDC,1h,2025-01-01 00:00:00+00:00,2026-01-01 00:00:00+00:00,8760,8760,0,100.0
8,holdout,BTC/USDC,15m,2026-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,20352,20352,0,100.0
9,holdout,BTC/USDC,1h,2026-01-01 00:00:00+00:00,2026-08-01 00:00:00+00:00,5088,5088,0,100.0


### 6. Produce the data-quality decision

In [6]:
validity_failures = int(profile_df[[
    "duplicate_timestamps", "null_cells", "invalid_prices", "invalid_volume", "invalid_ohlc"
]].sum().sum())
timeframe_mismatches = int(consistency_df[[
    column for column in consistency_df.columns if column.endswith("_mismatches")
]].sum().sum())
minimum_split_coverage = float(split_coverage_df["coverage_pct"].min())
largest_gap = int(gaps_df["missing_candles"].max()) if not gaps_df.empty else 0

if validity_failures == 0 and timeframe_mismatches == 0 and minimum_split_coverage >= 99.5:
    decision = "READY FOR SEGMENTED RESEARCH"
else:
    decision = "NEEDS REVISION"

decision_summary = {
    "decision": decision,
    "validity_failures": validity_failures,
    "timeframe_mismatches": timeframe_mismatches,
    "minimum_split_coverage_pct": round(minimum_split_coverage, 6),
    "largest_historical_gap_candles": largest_gap,
    "as_of_utc": max(frame["date"].max() for frame in frames.values()).isoformat(),
}

(REPORT_DIR / "data_quality_decision.json").write_text(
    json.dumps(decision_summary, indent=2), encoding="utf-8"
)
decision_summary

{'decision': 'READY FOR SEGMENTED RESEARCH',
 'validity_failures': 0,
 'timeframe_mismatches': 0,
 'minimum_split_coverage_pct': 100.0,
 'largest_historical_gap_candles': 15758,
 'as_of_utc': '2026-08-03T18:00:00+00:00'}

## Takeaways

- Use only the pre-declared segmented windows beginning `2024-04-01`; do not backtest across the earlier long discontinuity.
- The intended grain and OHLC domain rules are clean, and stored hourly candles reconcile with complete 15-minute aggregates.
- Minimum split coverage is 100.0000%, which is adequate for candle-level research with explicit gap handling.
- Data quality supports research decisions, but does not validate strategy profitability or real-world fills.
